# テーマ6 CNNによる画像分類

## 目的
- 画像分類でCNNがどのように使われるかを体験する
- 画像がピクセルの集合であり、縦横の空間構造を持つことを確認する
- Digits dataset を用いて学習と予測を行う
- 正解例・誤分類画像を見て、モデルの限界を考える

## 注意
- このテンプレートは PyTorch を使います
- 環境に PyTorch がない場合は、教員側で事前セットアップが必要です


## CNNが画像に向いている理由

画像は、縦横に並んだピクセルの集合です。表形式データのように列を独立に見るだけでなく、近くのピクセル同士の関係が重要になります。

CNN は、画像の小さな領域にフィルタを当てて特徴を取り出します。畳み込みのイメージは次の式で表せます。

$$
S(i, j) = \sum_m \sum_n I(i+m, j+n)K(m,n)
$$

ここで、$I$ は画像、$K$ はフィルタ、$S$ は取り出された特徴です。フィルタは、線、角、濃淡の変化などを検出する役割を持ちます。


### Step 0: ライブラリを読み込む
PyTorch と補助ライブラリを読み込み、CNN 学習の準備をします。ここでデバイス確認も行います。

### Step 0: CNN 学習の流れを確認する
このノートブックでは、手書き数字データを使って PyTorch で CNN を学習します。通常の表形式データと違い、画像では近くのピクセルどうしの関係に注目する点を意識してください。


In [ ]:
# ===== Step 0: ライブラリ読み込み =====
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

### Step 1: 画像データを読み込む
Digits データセットを読み込み、画像の形とラベルの形を確認します。最初の画像を表示して、入力のイメージをつかみます。

#### 解説: 画像データの形

Digits データセットの画像は 8×8 ピクセルです。PyTorch の CNN に入力するため、形を次のように整えます。

$$
(データ数, チャンネル数, 高さ, 幅)
$$

白黒画像なのでチャンネル数は 1 です。カラー画像なら、通常は RGB の3チャンネルになります。


In [ ]:
# ===== Step 1: データ読み込み =====
# TODO: Digits dataset を読み込む
digits = load_digits()
images = digits.images / 16.0  # 画素値を 0 から 1 にそろえる
labels = digits.target

print('画像データの形 =', images.shape)
print('ラベルの形 =', labels.shape)

plt.figure(figsize=(4, 4))
plt.imshow(images[0], cmap='gray')
plt.title(f'サンプル画像: label={labels[0]}')
plt.axis('off')
plt.show()

In [ ]:
# ===== 追加演習: 複数の数字を並べて見る =====
fig, axes = plt.subplots(2, 5, figsize=(8, 4))
for ax, image, label in zip(axes.ravel(), images[:10], labels[:10]):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'label={label}')
    ax.axis('off')
plt.tight_layout()
plt.show()


### Step 2: 学習用データを作る
画像データを train と test に分けて、PyTorch の Tensor と DataLoader に変換します。CNN に渡せる形へ整える工程です。

In [ ]:
# ===== Step 2: 学習用データ作成 =====
# TODO: train と test に分け、PyTorch 用 Tensor に変換する
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=64)

print('train batch 数 =', len(train_loader))

### Step 3: CNN モデルを定義する
畳み込み、ReLU、プーリング、全結合層を組み合わせた簡単な CNN を定義します。画像の一部分に注目して特徴を取り出す流れを確認してください。


#### 解説: プーリング

`MaxPool2d(2)` は、2×2 の範囲から最大値を取り出して画像を小さくします。これにより、位置が少しずれても似た特徴として扱いやすくなり、計算量も減ります。

このモデルでは、8×8 の画像が畳み込みとプーリングを通って、最後に分類用の全結合層へ渡されます。


In [ ]:
# ===== Step 3: CNNモデル =====
# TODO: CNN の構造を読んで、入力と出力の形を確認する
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 2 * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print(model)

### Step 4: モデルを学習する
定義した CNN を学習ループに通して、訓練を進めます。epoch ごとの loss を見て、学習が安定しているか確認します。

In [ ]:
# ===== Step 4: 学習 =====
# TODO: epoch 数を変えて結果を比較してもよい
epochs = 8

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch_images)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch + 1}/{epochs}, loss={total_loss / len(train_loader):.4f}')

### Step 5: 学習結果を確認する
テストデータで予測を行い、Accuracy と誤分類数を確認します。最後に誤分類画像を見て、数字の形が曖昧だったか、他の数字と似ていたかを考えます。


In [ ]:
# ===== Step 5: 評価と誤分類の可視化 =====
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        outputs = model(batch_images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch_labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
accuracy = (all_preds == all_labels).mean()
print(f'Accuracy: {accuracy:.3f}')

wrong_index = np.where(all_preds != all_labels)[0]
print('誤分類数 =', len(wrong_index))

if len(wrong_index) > 0:
    idx = wrong_index[0]  # TODO: 別の誤分類例も見てみる
    plt.figure(figsize=(3, 3))
    plt.imshow(X_test[idx], cmap='gray')
    plt.title(f'true={all_labels[idx]}, pred={all_preds[idx]}')
    plt.axis('off')
    plt.show()

#### 追加観察: 正解例と誤分類例を比べる

誤分類画像だけを見るのではなく、正しく分類できた画像とも比べてください。

- 線が薄い、太い、途切れている
- 他の数字と形が似ている
- 書き方に個人差がある
- 8×8 なので細かい情報が失われている

モデルの限界は、データの難しさとセットで考えると説明しやすくなります。


In [ ]:
# ===== 追加演習: 正解例を1つ表示する =====
correct_index = np.where(all_preds == all_labels)[0]
if len(correct_index) > 0:
    idx = correct_index[0]
    plt.figure(figsize=(3, 3))
    plt.imshow(X_test[idx], cmap='gray')
    plt.title(f'true={all_labels[idx]}, pred={all_preds[idx]}')
    plt.axis('off')
    plt.show()


### Step 5: 学習結果を確認した後に考えること
Accuracy だけでなく、正しく分類できた画像と間違えた画像の違いを見ます。誤分類例は、モデルが苦手なケースを考える手がかりになります。


## 振り返り
- CNN が画像分類に使われる理由を、自分の言葉で説明してください。
- 誤分類画像を見ると、なぜ間違えたと思いますか。
- テーマ5のニューラルネットワークと比べて、CNN では何が画像向けになっていると感じましたか。
